# Notebook 4: Business Insights

Dataset: McAuley-Lab/Amazon-Reviews-2023 — Pet Supplies

This notebook derives two focused business insights from the RoBERTa sentiment
predictions produced in Notebook 3:

1. **Complaint Urgency** — which negative reviews are most critical and require
   immediate action, identified by crossing sentiment labels with helpful vote scores
2. **Product-Specific Negative Keywords** — which product-related terms appear most
   frequently in negative reviews, surfacing recurring quality issues for product teams

Do not rerun Notebooks 1, 2, or 3 before this — it reads directly from
`data/cleaned_reviews_with_preds.csv`.

## 1. Imports & Configuration

In [ ]:
import warnings
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

PREDS_PATH = Path("data/cleaned_reviews_with_preds.csv")

COLOURS = {"Positive": "#2ECC71", "Neutral": "#F39C12", "Negative": "#E74C3C"}


## 2. Load Predictions

In [ ]:
df = pd.read_csv(PREDS_PATH)
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

print(f"Total reviews : {len(df):,}")
print()
print("RoBERTa sentiment distribution:")
print(df["roberta_pred_label"].value_counts().to_string())
print()
print("Columns available:")
print(df.columns.tolist())


## 3. Insight 1 — Complaint Urgency

**Business question:** Which negative reviews represent the most urgent complaints
that a product or customer experience team should act on first?

**Method:** Negative reviews are cross-referenced with helpful vote scores.
A helpful vote indicates that other customers found a complaint credible and
useful — it functions as a crowd-validation signal. Negative reviews with high
helpful votes are therefore the most business-critical, as they represent
widespread rather than isolated dissatisfaction.

### 3.1 Define Priority Tiers

In [ ]:
# Helpful vote threshold: top 25% of all reviews = high helpfulness
hv_threshold = df["helpful_vote"].quantile(0.75)
print(f"Helpful vote threshold (p75) : {hv_threshold:.0f}")

def assign_priority(row):
    """
    Three-tier priority framework:
    - CRITICAL : Negative sentiment + high helpful votes
                 → crowd-validated complaint, immediate QA escalation required
    - MONITOR  : Negative sentiment + low helpful votes
                 → individual complaint, review weekly for recurring patterns
    - ROUTINE  : Neutral or Positive sentiment
                 → no immediate action required
    """
    if row["roberta_pred_label"] == "Negative" and row["helpful_vote"] >= hv_threshold:
        return "Critical"
    elif row["roberta_pred_label"] == "Negative":
        return "Monitor"
    else:
        return "Routine"

df["priority"] = df.apply(assign_priority, axis=1)

priority_counts = df["priority"].value_counts()
print()
print("Priority tier distribution:")
print(priority_counts.to_string())


### 3.2 Priority Distribution Chart

In [ ]:
tier_colours = {"Critical": "#E74C3C", "Monitor": "#F39C12", "Routine": "#2ECC71"}
tiers        = ["Critical", "Monitor", "Routine"]
counts       = [priority_counts.get(t, 0) for t in tiers]
pcts         = [c / len(df) * 100 for c in counts]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tiers, counts,
              color=[tier_colours[t] for t in tiers],
              edgecolor="white", linewidth=0.6, width=0.5)

for bar, count, pct in zip(bars, counts, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.01,
            f"{count:,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=10)

ax.set_title("Complaint Urgency — Priority Tier Distribution",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Reviews")
ax.set_ylim(0, max(counts) * 1.18)
plt.tight_layout()
plt.savefig("data/fig_priority_tiers.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.3 Confidence Distribution by Priority Tier

In [ ]:
# Reviews classified as Critical should show high model confidence.
# Low confidence in the Critical tier would indicate the model is uncertain
# about those flagged complaints — an important limitation to report.
fig, ax = plt.subplots(figsize=(9, 4))

for tier, colour in tier_colours.items():
    subset = df[df["priority"] == tier]["roberta_confidence"]
    if len(subset) > 0:
        ax.hist(subset, bins=20, alpha=0.6, label=tier,
                color=colour, edgecolor="white", linewidth=0.3)

ax.set_title("RoBERTa Confidence by Priority Tier",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Prediction Confidence (max softmax probability)")
ax.set_ylabel("Count")
ax.legend(title="Priority Tier")
plt.tight_layout()
plt.savefig("data/fig_priority_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

# Mean confidence per tier
print("Mean RoBERTa confidence per tier:")
print(df.groupby("priority")["roberta_confidence"].mean().round(3).to_string())


### 3.4 Brand Response Framework

In [ ]:
framework = pd.DataFrame({
    "Tier"    : ["Critical", "Monitor", "Routine"],
    "Criteria": [
        "Negative sentiment + top 25% helpful votes",
        "Negative sentiment + below threshold helpful votes",
        "Neutral or Positive sentiment",
    ],
    "Team"    : ["Product Safety / QA", "CX / Product Dev", "Brand / Marketing"],
    "SLA"     : ["24 hours", "Weekly review", "Monthly review"],
    "Action"  : [
        "Immediate product investigation; escalate to QA and supply chain",
        "Aggregate weekly; identify recurring complaint themes",
        "Monitor for shifts; extract satisfaction drivers for marketing",
    ],
})

print("BRAND RESPONSE FRAMEWORK")
print("=" * 90)
print(framework.to_string(index=False))
framework.to_csv("data/brand_response_framework.csv", index=False)
print("\nFramework saved → data/brand_response_framework.csv")


### 3.5 Sample Critical Reviews

In [ ]:
critical = (df[df["priority"] == "Critical"]
            .sort_values("helpful_vote", ascending=False)
            [["rating", "review_text", "helpful_vote", "roberta_confidence"]]
            .head(10))

print(f"Top 10 Critical reviews by helpful vote count:")
print(critical.to_string(index=False))
critical.to_csv("data/critical_reviews_sample.csv", index=False)


## 4. Insight 2 — Product-Specific Negative Keywords

**Business question:** Which product-related words appear most frequently in
negative reviews, and what recurring quality issues do they point to?

**Method:** All reviews classified as Negative by RoBERTa are filtered and their
text is tokenised. Common stopwords are removed, leaving only content words that
carry product-specific meaning. The resulting frequency distribution surfaces
recurring complaint themes that product and supply chain teams can act on.

### 4.1 Extract Keywords from Negative Reviews

In [ ]:
# Expanded stopword list tuned for Amazon review language.
# Generic sentiment words (bad, good, love) are excluded because they describe
# sentiment rather than product attributes — the goal is product-specific signals.
STOPWORDS = set([
    "the","a","an","i","it","is","was","to","in","and","of","for","my","this",
    "that","with","but","not","have","had","they","be","on","at","or","so","as",
    "very","just","from","up","do","are","its","we","me","got","get","would",
    "could","has","one","if","when","after","about","than","more","also","only",
    "now","out","like","her","him","their","them","our","your","you","he","she",
    "no","will","all","been","were","what","then","can","did","there","his",
    "which","other","some","into","over","those","too","even","still","back",
    "purchased","bought","ordered","amazon","review","product","item","buy",
    "good","bad","great","love","hate","nice","well","really","use","used",
    "using","time","day","week","month","year","said","went","come","came",
    "think","thought","know","see","look","try","tried","need","want","make",
    "made","give","gave","take","took","new","old","little","big","small",
    "much","many","every","never","always","already","still","however","though"
])

def extract_keywords(texts, stopwords=STOPWORDS, min_length=3, top_n=30):
    """
    Tokenise texts, filter stopwords and short tokens, return top_n by frequency.
    min_length=3 removes uninformative short tokens (e.g. 'ok', 'it').
    """
    words = []
    for t in texts:
        tokens = re.findall(r"[a-z]{%d,}" % min_length, t.lower())
        words.extend([w for w in tokens if w not in stopwords])
    return Counter(words).most_common(top_n)

neg_reviews  = df[df["roberta_pred_label"] == "Negative"]["review_text"].fillna("")
neg_keywords = extract_keywords(neg_reviews, top_n=30)

print(f"Negative reviews analysed : {len(neg_reviews):,}")
print()
print("Top 30 keywords in negative reviews:")
for rank, (word, count) in enumerate(neg_keywords, 1):
    print(f"  {rank:>2}. {word:<20} {count:>4}")


### 4.2 Keyword Frequency Chart

In [ ]:
words, counts = zip(*neg_keywords[:25])

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(words[::-1], counts[::-1],
               color="#E74C3C", edgecolor="white", linewidth=0.4)

for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + max(counts) * 0.005,
            bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=8)

ax.set_title("Top 25 Product-Specific Keywords in Negative Reviews",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Frequency")
ax.set_xlim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig("data/fig_neg_keywords.png", dpi=150, bbox_inches="tight")
plt.show()


### 4.3 Keyword Comparison: Negative vs. Positive Reviews

In [ ]:
# Comparing keyword frequency between negative and positive reviews reveals
# which terms are genuinely complaint-specific vs. terms that appear in all
# reviews regardless of sentiment (e.g. generic product category words).
pos_reviews  = df[df["roberta_pred_label"] == "Positive"]["review_text"].fillna("")
pos_keywords = dict(extract_keywords(pos_reviews, top_n=50))
neg_keywords_dict = dict(neg_keywords)

# Compute negative excess: how much more often a word appears in negative
# vs. positive reviews, normalised by review count
neg_n = len(neg_reviews)
pos_n = len(pos_reviews)

excess = {}
for word, neg_count in neg_keywords_dict.items():
    pos_count = pos_keywords.get(word, 0)
    neg_rate  = neg_count / max(neg_n, 1)
    pos_rate  = pos_count / max(pos_n, 1)
    excess[word] = neg_rate - pos_rate

# Top 15 words with highest negative excess — most complaint-specific
top_excess = sorted(excess.items(), key=lambda x: x[1], reverse=True)[:15]
exc_words, exc_vals = zip(*top_excess)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(exc_words[::-1], exc_vals[::-1],
        color="#C0392B", edgecolor="white", linewidth=0.4)
ax.set_title("Most Complaint-Specific Keywords
(Negative excess rate vs. Positive reviews)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Negative rate − Positive rate")
plt.tight_layout()
plt.savefig("data/fig_keyword_excess.png", dpi=150, bbox_inches="tight")
plt.show()

print("Most complaint-specific keywords (unique to negative reviews):")
for word, val in top_excess:
    print(f"  {word:<20} excess rate = {val:.4f}")

# Save keyword data
kw_df = pd.DataFrame(neg_keywords, columns=["keyword", "frequency"])
kw_df["pos_frequency"] = kw_df["keyword"].map(lambda w: pos_keywords.get(w, 0))
kw_df["neg_excess_rate"] = kw_df["keyword"].map(lambda w: excess.get(w, 0))
kw_df.to_csv("data/negative_keywords.csv", index=False)
print("\nKeyword data saved → data/negative_keywords.csv")


## 5. Combined Summary

In [ ]:
print("=" * 60)
print("BUSINESS INSIGHTS SUMMARY")
print("=" * 60)

print()
print("INSIGHT 1 — COMPLAINT URGENCY")
print("-" * 40)
for tier in ["Critical", "Monitor", "Routine"]:
    n   = (df["priority"] == tier).sum()
    pct = n / len(df) * 100
    print(f"  {tier:<10} : {n:>6,} reviews ({pct:.1f}%)")

critical_df = df[df["priority"] == "Critical"]
print()
print(f"  Critical reviews mean helpful votes : "
      f"{critical_df['helpful_vote'].mean():.2f}")
print(f"  Critical reviews mean confidence    : "
      f"{critical_df['roberta_confidence'].mean():.3f}")

print()
print("INSIGHT 2 — TOP 10 COMPLAINT KEYWORDS")
print("-" * 40)
for word, count in neg_keywords[:10]:
    print(f"  {word:<20} {count:>4} occurrences")

print()
print("All outputs saved to data/")
print("  fig_priority_tiers.png")
print("  fig_priority_confidence.png")
print("  fig_neg_keywords.png")
print("  fig_keyword_excess.png")
print("  brand_response_framework.csv")
print("  critical_reviews_sample.csv")
print("  negative_keywords.csv")
